# SmolVLM vs DINOv2 on Oxford-IIIT Pets

This is my final Kaggle notebook for comparing two ways to solve pet breed recognition on the Oxford-IIIT Pet dataset.

I wanted to test a simple question: for a fixed set of breed labels, is it better to use a dedicated image classifier, or can a small visual-language model get close if I fine-tune it to answer with the breed name?

The notebook trains and evaluates:

1. `facebook/dinov2-small` as a direct 37-class image classifier.
2. `HuggingFaceTB/SmolVLM-256M-Instruct` as a VQA-style model answering: `What breed is this pet?`
3. A shared set of exports: prediction CSVs, confusion matrices, summary JSON, and a final comparison table.

I kept the default sample sizes conservative so this can run on a free Kaggle GPU. Once the full notebook runs cleanly, the sample counts can be increased for a stronger rerun.


## Runtime Notes

I ran this as a Kaggle-style GPU notebook.

Before running from the top:

- Turn Internet on.
- Use a GPU accelerator, ideally T4 or P100.
- Restart the session after dependency installation if Kaggle keeps an old PIL or `torchao` package loaded.

The code below is intentionally a little defensive because Kaggle images change over time, and the PIL/Transformers import chain can break when package versions are mixed.


In [ ]:
# Keep installs in the first cell so Kaggle resolves package versions before imports.
!pip -q install -U --no-cache-dir \
    "transformers>=4.52.0" \
    "datasets>=3.0.0" \
    "accelerate>=0.34.0" \
    "peft>=0.15.0" \
    "torchao>=0.16.0" \
    "evaluate>=0.4.0" \
    scikit-learn matplotlib "pandas<3.0.0" tqdm

# Kaggle sometimes ships a stale PIL build. Reinstalling Pillow avoids the
# `_Ink` import error that can appear when Transformers imports image utilities.
!pip -q install --no-cache-dir --force-reinstall "pillow==11.3.0"


In [ ]:
import PIL
import transformers
from PIL import Image
from transformers import AutoProcessor, AutoModelForImageTextToText

print("Pillow:", PIL.__version__)
print("Transformers:", transformers.__version__)
print("SmolVLM auto classes import OK")


In [ ]:
import os
os.environ["TOKENIZERS_PARALLELISM"] = "false"
os.environ["WANDB_DISABLED"] = "true"

import gc
import re
import json
import random
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import matplotlib.pyplot as plt
from tqdm.auto import tqdm

from sklearn.metrics import accuracy_score, f1_score, classification_report, confusion_matrix, ConfusionMatrixDisplay

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

WORK_DIR = Path("/kaggle/working") if Path("/kaggle/working").exists() else Path("./working")
WORK_DIR.mkdir(parents=True, exist_ok=True)

print("Torch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    print("VRAM GB:", round(torch.cuda.get_device_properties(0).total_memory / 1024**3, 2))
else:
    raise RuntimeError("No GPU found. In Kaggle, enable GPU in Notebook Settings.")


def clear_memory():
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
        torch.cuda.ipc_collect()


In [ ]:
from datasets import load_dataset

DATASET_ID = "timm/oxford-iiit-pet"
raw = load_dataset(DATASET_ID)

print(raw)
print(raw["train"].features)
print(raw["train"][0].keys())


In [ ]:
# I keep the column detection explicit so the notebook fails early if the
# dataset schema changes.
sample0 = raw["train"][0]

IMAGE_COL = "image"
if IMAGE_COL not in sample0:
    raise ValueError(f"Expected an image column named {IMAGE_COL}. Found columns: {list(sample0.keys())}")

candidate_label_cols = ["label", "labels", "category", "class"]
LABEL_COL = None
for col in candidate_label_cols:
    if col in sample0:
        LABEL_COL = col
        break

if LABEL_COL is None:
    raise ValueError(f"Could not find a label column. Found columns: {list(sample0.keys())}")

label_feature = raw["train"].features[LABEL_COL]
if hasattr(label_feature, "names") and label_feature.names is not None:
    label_names = list(label_feature.names)
else:
    unique_labels = sorted(set(raw["train"][LABEL_COL]))
    label_names = [str(x) for x in unique_labels]

id2label = {i: name for i, name in enumerate(label_names)}
label2id = {name: i for i, name in enumerate(label_names)}
num_labels = len(label_names)

print("Image column:", IMAGE_COL)
print("Label column:", LABEL_COL)
print("Num labels:", num_labels)
print(label_names[:10])


In [ ]:
# These are the sample sizes used for the final exported run. I kept DINO on a
# larger test split because it is cheaper to evaluate than the generative VLM.
DINO_TRAIN_N = 1200
DINO_EVAL_N = 300
DINO_TEST_N = 500

VLM_TRAIN_N = 600
VLM_EVAL_N = 120
VLM_TEST_N = 200

train_raw_dino = raw["train"].shuffle(seed=SEED).select(range(min(DINO_TRAIN_N, len(raw["train"]))))
eval_raw_dino = raw["test"].shuffle(seed=SEED).select(range(min(DINO_EVAL_N, len(raw["test"]))))
test_raw_dino = raw["test"].shuffle(seed=SEED + 1).select(range(min(DINO_TEST_N, len(raw["test"]))))

train_raw_vlm = raw["train"].shuffle(seed=SEED).select(range(min(VLM_TRAIN_N, len(raw["train"]))))
eval_raw_vlm = raw["test"].shuffle(seed=SEED).select(range(min(VLM_EVAL_N, len(raw["test"]))))
test_raw_vlm = raw["test"].shuffle(seed=SEED + 1).select(range(min(VLM_TEST_N, len(raw["test"]))))

print("DINO splits:", len(train_raw_dino), len(eval_raw_dino), len(test_raw_dino))
print("VLM splits:", len(train_raw_vlm), len(eval_raw_vlm), len(test_raw_vlm))


## Part A: DINOv2 Classifier

I start with DINOv2 because it is the clean baseline for this problem: one image in, one breed label out. Since Oxford-IIIT Pets has a fixed label set, this is the model I expected to be strongest.


In [ ]:
from transformers import AutoImageProcessor, AutoModelForImageClassification, TrainingArguments, Trainer

DINO_MODEL_ID = "facebook/dinov2-small"

dino_processor = AutoImageProcessor.from_pretrained(DINO_MODEL_ID)
dino_model = AutoModelForImageClassification.from_pretrained(
    DINO_MODEL_ID,
    num_labels=num_labels,
    id2label=id2label,
    label2id=label2id,
    ignore_mismatched_sizes=True,
)

print("Loaded", DINO_MODEL_ID)

In [ ]:
def dino_collate_fn(batch):
    images = [x[IMAGE_COL].convert("RGB") for x in batch]
    labels = torch.tensor([int(x[LABEL_COL]) for x in batch], dtype=torch.long)
    enc = dino_processor(images=images, return_tensors="pt")
    enc["labels"] = labels
    return enc

def compute_dino_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    return {
        "accuracy": accuracy_score(labels, preds),
        "macro_f1": f1_score(labels, preds, average="macro"),
    }

In [ ]:
DINO_OUTPUT_DIR = str(WORK_DIR / "dinov2-oxford-pets")

dino_args = TrainingArguments(
    output_dir=DINO_OUTPUT_DIR,
    num_train_epochs=20,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    learning_rate=5e-5,
    warmup_ratio=0.05,
    eval_strategy="epoch",
    save_strategy="epoch",
    save_total_limit=1,
    load_best_model_at_end=True,
    metric_for_best_model="accuracy",
    greater_is_better=True,
    fp16=True,
    logging_steps=20,
    report_to="none",
    remove_unused_columns=False,
)

dino_trainer = Trainer(
    model=dino_model,
    args=dino_args,
    train_dataset=train_raw_dino,
    eval_dataset=eval_raw_dino,
    data_collator=dino_collate_fn,
    compute_metrics=compute_dino_metrics,
)

print("DINO Trainer ready")

In [ ]:
dino_train_result = dino_trainer.train()
print(dino_train_result)

dino_eval_metrics = dino_trainer.evaluate()
print("DINO eval:", dino_eval_metrics)

dino_trainer.save_model(DINO_OUTPUT_DIR)
dino_processor.save_pretrained(DINO_OUTPUT_DIR)
print("Saved DINO model to", DINO_OUTPUT_DIR)

In [ ]:
dino_pred_output = dino_trainer.predict(test_raw_dino)
dino_logits = dino_pred_output.predictions
dino_y_true = dino_pred_output.label_ids
dino_y_pred = np.argmax(dino_logits, axis=-1)

dino_test_accuracy = accuracy_score(dino_y_true, dino_y_pred)
dino_test_macro_f1 = f1_score(dino_y_true, dino_y_pred, average="macro")

print("DINO test accuracy:", dino_test_accuracy)
print("DINO test macro-F1:", dino_test_macro_f1)

rows = []
for i, (yt, yp) in enumerate(zip(dino_y_true, dino_y_pred)):
    rows.append({
        "idx": i,
        "true_id": int(yt),
        "pred_id": int(yp),
        "true_label": id2label[int(yt)].replace("_", " ").lower(),
        "pred_label": id2label[int(yp)].replace("_", " ").lower(),
        "correct": bool(int(yt) == int(yp)),
    })

dino_results = pd.DataFrame(rows)
dino_results.to_csv(WORK_DIR / "dino_predictions.csv", index=False)
display(dino_results.head(20))


In [ ]:
dino_labels_present = sorted(set(dino_y_true).union(set(dino_y_pred)))
dino_display_labels = [id2label[i].replace("_", " ") for i in dino_labels_present]
dino_cm = confusion_matrix(dino_y_true, dino_y_pred, labels=dino_labels_present)

fig, ax = plt.subplots(figsize=(14, 14))
disp = ConfusionMatrixDisplay(confusion_matrix=dino_cm, display_labels=dino_display_labels)
disp.plot(ax=ax, xticks_rotation=90, values_format="d", colorbar=False)
plt.title("DINOv2 Oxford Pets Confusion Matrix")
plt.tight_layout()
plt.savefig(WORK_DIR / "dino_confusion_matrix.png", dpi=160)
plt.show()


In [ ]:
# Free GPU memory before loading the VLM. This matters on T4 sessions.
del dino_model, dino_trainer
clear_memory()


## Part B: SmolVLM Breed VQA

For SmolVLM, I frame the same classification problem as a visual question answering task:

- User: image + `What breed is this pet?`
- Assistant: breed name only

The image placeholder in the chat template is important. The message contains `{ "type": "image" }`, and the actual PIL image is passed separately to the processor. Without that placeholder, SmolVLM can raise an image-count mismatch error.


In [ ]:
SYSTEM_MESSAGE = (
    "You are a visual recognition assistant. "
    "Answer the user's question using only the image. "
    "For breed questions, answer with only the breed name and no extra words."
)

QUESTION = "What breed is this pet?"

def clean_breed_name(x):
    if isinstance(x, (int, np.integer)):
        name = id2label[int(x)]
    else:
        name = str(x)
    return name.replace("_", " ").lower().strip()

def format_vlm_example(sample):
    image = sample[IMAGE_COL].convert("RGB")
    answer = clean_breed_name(sample[LABEL_COL])
    return {
        "images": [image],
        "messages": [
            {
                "role": "system",
                "content": [{"type": "text", "text": SYSTEM_MESSAGE}],
            },
            {
                "role": "user",
                "content": [
                    {"type": "image"},
                    {"type": "text", "text": QUESTION},
                ],
            },
            {
                "role": "assistant",
                "content": [{"type": "text", "text": answer}],
            },
        ],
        "answer": answer,
    }

train_dataset_vlm = [format_vlm_example(x) for x in train_raw_vlm]
eval_dataset_vlm = [format_vlm_example(x) for x in eval_raw_vlm]
test_dataset_vlm = [format_vlm_example(x) for x in test_raw_vlm]

print(train_dataset_vlm[0]["messages"])
train_dataset_vlm[0]["images"][0]

In [ ]:
from transformers import AutoProcessor, AutoModelForImageTextToText
from peft import LoraConfig, get_peft_model

# 256M is the safest default for free Kaggle GPU. Change to 500M after the pipeline works.
VLM_MODEL_ID = "HuggingFaceTB/SmolVLM-256M-Instruct"
# VLM_MODEL_ID = "HuggingFaceTB/SmolVLM-500M-Instruct"  # optional upgrade

DTYPE = torch.float16
ATTN_IMPL = "eager"  # safer than flash-attn on Kaggle

vlm_processor = AutoProcessor.from_pretrained(
    VLM_MODEL_ID,
    size={"longest_edge": 384},
)

vlm_model = AutoModelForImageTextToText.from_pretrained(
    VLM_MODEL_ID,
    device_map="auto",
    torch_dtype=DTYPE,
    _attn_implementation=ATTN_IMPL,
)

vlm_model.config.use_cache = False
if hasattr(vlm_model, "gradient_checkpointing_enable"):
    vlm_model.gradient_checkpointing_enable()

print("Loaded", VLM_MODEL_ID)

In [ ]:
# LoRA keeps the VLM experiment small enough for a Kaggle GPU while still
# letting the language/vision bridge adapt to breed-name answers.
vlm_lora_config = LoraConfig(
    r=8,
    lora_alpha=16,
    lora_dropout=0.05,
    target_modules=[
        "q_proj", "k_proj", "v_proj", "o_proj",
        "gate_proj", "up_proj", "down_proj",
    ],
    bias="none",
    task_type="CAUSAL_LM",
)

vlm_model = get_peft_model(vlm_model, vlm_lora_config)
vlm_model.print_trainable_parameters()


In [ ]:
# Quick check for the image-placeholder issue before spending time training.
_smoke_messages = train_dataset_vlm[0]["messages"][:2]
_smoke_prompt = vlm_processor.apply_chat_template(
    _smoke_messages,
    add_generation_prompt=True,
)
print("Smoke prompt preview:")
print(repr(_smoke_prompt[:500]))

_smoke_inputs = vlm_processor(
    text=_smoke_prompt,
    images=train_dataset_vlm[0]["images"],
    return_tensors="pt",
)
print("Smoke test OK. Keys:", list(_smoke_inputs.keys()))
print("input_ids shape:", _smoke_inputs["input_ids"].shape)


In [ ]:
class VLMListDataset(torch.utils.data.Dataset):
    def __init__(self, examples):
        self.examples = examples
    def __len__(self):
        return len(self.examples)
    def __getitem__(self, idx):
        return self.examples[idx]

vlm_train_ds = VLMListDataset(train_dataset_vlm)
vlm_eval_ds = VLMListDataset(eval_dataset_vlm)

MAX_LENGTH = 512

def vlm_collate_fn(batch):
    texts = []
    images = []
    for ex in batch:
        # Full conversation includes the assistant answer for supervised fine-tuning.
        text = vlm_processor.apply_chat_template(
            ex["messages"],
            add_generation_prompt=False,
        )
        texts.append(text)
        images.append(ex["images"][0].convert("RGB"))

    enc = vlm_processor(
        text=texts,
        images=images,
        return_tensors="pt",
        padding=True,
        truncation=True,
        max_length=MAX_LENGTH,
    )

    labels = enc["input_ids"].clone()
    pad_token_id = vlm_processor.tokenizer.pad_token_id
    if pad_token_id is not None:
        labels[labels == pad_token_id] = -100
    enc["labels"] = labels
    return enc

# Collator smoke test.
_batch = vlm_collate_fn([vlm_train_ds[0], vlm_train_ds[1]])
print({k: tuple(v.shape) if hasattr(v, "shape") else type(v) for k, v in _batch.items()})

In [ ]:
VLM_OUTPUT_DIR = str(WORK_DIR / "smolvlm-oxford-pets-lora")

vlm_args = TrainingArguments(
    output_dir=VLM_OUTPUT_DIR,
    num_train_epochs=20,
    per_device_train_batch_size=2,
    per_device_eval_batch_size=2,
    gradient_accumulation_steps=8,
    learning_rate=5e-5,
    weight_decay=0.01,
    warmup_steps=55,
    eval_strategy="epoch",
    save_strategy="epoch",
    save_total_limit=2,
    load_best_model_at_end=True,
    fp16=True,
    logging_steps=10,
    report_to="none",
    remove_unused_columns=False,
    gradient_checkpointing=True,
)

vlm_trainer = Trainer(
    model=vlm_model,
    args=vlm_args,
    train_dataset=vlm_train_ds,
    eval_dataset=vlm_eval_ds,
    data_collator=vlm_collate_fn,
)

print("SmolVLM Trainer ready")


In [ ]:
vlm_train_result = vlm_trainer.train()
print(vlm_train_result)

vlm_trainer.save_model(VLM_OUTPUT_DIR)
vlm_processor.save_pretrained(VLM_OUTPUT_DIR)
print("Saved SmolVLM LoRA adapter and processor to", VLM_OUTPUT_DIR)

In [ ]:
def normalize_text(s):
    s = str(s).lower().strip()
    s = re.sub(r"[^a-z0-9 ]+", " ", s)
    s = re.sub(r"\s+", " ", s)
    return s.strip()


known_breeds = [normalize_text(x.replace("_", " ")) for x in label_names]


def map_to_known_breed(pred):
    pred = normalize_text(pred)

    # Try exact and substring matches first. This handles answers like
    # "this looks like a shiba inu" without rewarding unrelated text too much.
    for breed in known_breeds:
        if pred == breed or breed in pred:
            return breed

    pred_tokens = set(pred.split())
    if not pred_tokens:
        return "unknown"

    best_breed, best_score = "unknown", 0
    for breed in known_breeds:
        score = len(pred_tokens.intersection(set(breed.split())))
        if score > best_score:
            best_breed, best_score = breed, score

    return best_breed if best_score > 0 else "unknown"


def generate_vlm_answer(sample, max_new_tokens=20):
    vlm_model.eval()
    messages = sample["messages"][:2]
    prompt = vlm_processor.apply_chat_template(messages, add_generation_prompt=True)
    image = sample["images"][0].convert("RGB")
    inputs = vlm_processor(text=prompt, images=[image], return_tensors="pt").to(vlm_model.device)

    with torch.no_grad():
        generated_ids = vlm_model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False,
        )

    input_len = inputs["input_ids"].shape[1]
    new_tokens = generated_ids[:, input_len:]
    text = vlm_processor.batch_decode(new_tokens, skip_special_tokens=True)[0]
    return text.strip()


print("True:", test_dataset_vlm[0]["answer"])
print("Pred:", generate_vlm_answer(test_dataset_vlm[0]))


In [ ]:
# SmolVLM test evaluation.
vlm_rows = []
for i, sample in enumerate(tqdm(test_dataset_vlm)):
    true_answer = normalize_text(sample["answer"])
    pred_answer = normalize_text(generate_vlm_answer(sample))
    pred_class = map_to_known_breed(pred_answer)

    vlm_rows.append({
        "idx": i,
        "true": true_answer,
        "raw_pred": pred_answer,
        "pred_class": pred_class,
        "text_match_correct": bool(true_answer == pred_answer or true_answer in pred_answer),
        "class_correct": bool(true_answer == pred_class),
    })

vlm_results = pd.DataFrame(vlm_rows)
vlm_results.to_csv(WORK_DIR / "smolvlm_predictions.csv", index=False)
display(vlm_results.head(20))

vlm_text_match_accuracy = float(vlm_results["text_match_correct"].mean())
vlm_class_accuracy = float(vlm_results["class_correct"].mean())
vlm_macro_f1 = f1_score(vlm_results["true"], vlm_results["pred_class"], average="macro", zero_division=0)

print("SmolVLM text-match accuracy:", vlm_text_match_accuracy)
print("SmolVLM mapped-class accuracy:", vlm_class_accuracy)
print("SmolVLM mapped-class macro-F1:", vlm_macro_f1)
print(classification_report(vlm_results["true"], vlm_results["pred_class"], zero_division=0))

In [ ]:
# SmolVLM confusion matrix.
vlm_present_classes = sorted(set(vlm_results["true"]).union(set(vlm_results["pred_class"])))
vlm_present_classes_no_unknown = [c for c in vlm_present_classes if c != "unknown"]

vlm_cm = confusion_matrix(
    vlm_results["true"],
    vlm_results["pred_class"],
    labels=vlm_present_classes_no_unknown,
)

fig, ax = plt.subplots(figsize=(14, 14))
disp = ConfusionMatrixDisplay(confusion_matrix=vlm_cm, display_labels=vlm_present_classes_no_unknown)
disp.plot(ax=ax, xticks_rotation=90, values_format="d", colorbar=False)
plt.title("SmolVLM Oxford Pets Mapped-Breed Confusion Matrix")
plt.tight_layout()
plt.savefig(WORK_DIR / "smolvlm_confusion_matrix.png", dpi=160)
plt.show()

## Part C: Final Comparison

At this point both models have been evaluated on their held-out test subsets. I save the side-by-side table so the notebook output can be used directly in the report and repository README.


In [ ]:
comparison = pd.DataFrame([
    {
        "model": "DINOv2-small",
        "task": "closed-set image classification",
        "train_samples": len(train_raw_dino),
        "test_samples": len(test_raw_dino),
        "accuracy": float(dino_test_accuracy),
        "macro_f1": float(dino_test_macro_f1),
        "notes": "Direct classifier; expected to be stronger for fixed breed labels.",
    },
    {
        "model": VLM_MODEL_ID,
        "task": "image + question -> breed text",
        "train_samples": len(train_raw_vlm),
        "test_samples": len(test_raw_vlm),
        "accuracy": float(vlm_class_accuracy),
        "macro_f1": float(vlm_macro_f1),
        "notes": "Fine-tuned VLM; flexible natural-language output, mapped back to labels.",
    },
])

comparison.to_csv(WORK_DIR / "model_comparison.csv", index=False)
display(comparison)

In [ ]:
# Bar plot comparison.
fig, ax = plt.subplots(figsize=(8, 5))
ax.bar(comparison["model"], comparison["accuracy"])
ax.set_ylim(0, 1)
ax.set_ylabel("Accuracy")
ax.set_title("DINOv2 vs SmolVLM on Oxford-IIIT Pet Breed Recognition")
plt.xticks(rotation=20, ha="right")
plt.tight_layout()
plt.savefig(WORK_DIR / "comparison_accuracy.png", dpi=160)
plt.show()

In [ ]:
# Save summary and zip outputs.
summary = {
    "dataset": DATASET_ID,
    "dino_model": DINO_MODEL_ID,
    "vlm_model": VLM_MODEL_ID,
    "dino_train_samples": len(train_raw_dino),
    "dino_test_samples": len(test_raw_dino),
    "vlm_train_samples": len(train_raw_vlm),
    "vlm_test_samples": len(test_raw_vlm),
    "dino_test_accuracy": float(dino_test_accuracy),
    "dino_test_macro_f1": float(dino_test_macro_f1),
    "vlm_text_match_accuracy": float(vlm_text_match_accuracy),
    "vlm_mapped_class_accuracy": float(vlm_class_accuracy),
    "vlm_mapped_class_macro_f1": float(vlm_macro_f1),
}

with open(WORK_DIR / "summary.json", "w") as f:
    json.dump(summary, f, indent=2)

print(json.dumps(summary, indent=2))
print("Saved outputs in", WORK_DIR)

In [ ]:
# Bundle the outputs that matter for review. The trained model folders are
# included because this was a Kaggle run, but they can be removed if the zip is
# only meant to store metrics and plots.
!zip -r final_smolvlm_dino_pet_project_outputs.zip     {WORK_DIR}/dino_predictions.csv     {WORK_DIR}/smolvlm_predictions.csv     {WORK_DIR}/model_comparison.csv     {WORK_DIR}/summary.json     {WORK_DIR}/dino_confusion_matrix.png     {WORK_DIR}/smolvlm_confusion_matrix.png     {WORK_DIR}/comparison_accuracy.png     {WORK_DIR}/smolvlm-oxford-pets-lora     {WORK_DIR}/dinov2-oxford-pets


## What I Would Try Next

The first improvement is to make the comparison more controlled by evaluating both models on the exact same test subset size. After that, I would scale the run like this:

```python
DINO_TRAIN_N = 3000
DINO_EVAL_N = 700
DINO_TEST_N = 1000
VLM_TRAIN_N = 1500
VLM_EVAL_N = 300
VLM_TEST_N = 500
```

For the VLM side, I would also try `HuggingFaceTB/SmolVLM-500M-Instruct` and enforce a stricter output format so the model returns only one breed label. DINOv2 should still be stronger for fixed-label classification, but SmolVLM is useful when the final product needs natural-language visual QA rather than only a class ID.
